# Pre-procesamiento WSI — Extracción de Patches (Local)



## Celda 1 — Importar librerías

In [1]:
import os
import numpy as np
import tiatoolbox
from PIL import Image
from pathlib import Path
from tiatoolbox.wsicore.wsireader import WSIReader
from tiatoolbox.tools.tissuemask import OtsuTissueMasker
from tiatoolbox.tools.patchextraction import get_patch_extractor

print('Librerías importadas correctamente')
print(f' NumPy     : {np.__version__}')
print(f' tiatoolbox: {tiatoolbox.__version__}')

Librerías importadas correctamente
 NumPy     : 2.0.0
 tiatoolbox: 2.0.1


## Celda 2 — Configurar rutas y parámetros

In [4]:
# ─── EDITÁ ESTAS RUTAS ─────────────────────────────────────────────────────

# Directorio donde están las láminas .ndpi
# Las láminas deben llamarse: lam_1.ndpi, lam_2.ndpi, lam_3.ndpi, ...
DIR_LAMINAS = 'C:/Users/admin2/Desktop/Fuentes_tesis/laminas/'
DIR_SALIDA  = 'C:/Users/admin2/Desktop/Fuentes_tesis/parches/'

# ─── PARÁMETROS DE EXTRACCIÓN ──────────────────────────────────────────────
PATCH_SIZE     = (2048, 1024)  # ancho x alto en píxeles
RESOLUTION     = 20            # magnificación de extracción
UNITS          = 'power'       # unidades de resolución
STRIDE         = (2048, 1024)  # desplazamiento (sin solapamiento)
MIN_MASK_RATIO = 0.8           # mínimo 80% de tejido por patch
MAX_PATCHES    = 20            # límite de patches por lámina
PNG_COMPRESS   = 9             # compresión PNG: 0 (sin compresión) a 9 (máxima, lossless)
# ───────────────────────────────────────────────────────────────────────────

# Verificar que el directorio de láminas existe
assert os.path.exists(DIR_LAMINAS), f' Directorio no encontrado: {DIR_LAMINAS}'

# Crear directorio de salida si no existe
os.makedirs(DIR_SALIDA, exist_ok=True)

# Detectar láminas disponibles en el directorio
laminas = sorted([
    f for f in os.listdir(DIR_LAMINAS)
    if f.endswith('.ndpi') and f.startswith('lam_')
])

if not laminas:
    print(' No se encontraron láminas .ndpi en el directorio')
    print(f'   Verificá que los archivos se llamen: lam_1.ndpi, lam_2.ndpi, ...')
    print(f'   Directorio revisado: {DIR_LAMINAS}')
else:
    print(f' Directorio de láminas : {DIR_LAMINAS}')
    print(f' Directorio de salida  : {DIR_SALIDA}')
    print(f'\n Láminas encontradas ({len(laminas)}):')
    for lam in laminas:
        print(f'   → {lam}')

 Directorio de láminas : C:/Users/admin2/Desktop/Fuentes_tesis/laminas/
 Directorio de salida  : C:/Users/admin2/Desktop/Fuentes_tesis/parches/

 Láminas encontradas (26):
   → lam_36.ndpi
   → lam_37.ndpi
   → lam_38.ndpi
   → lam_39.ndpi
   → lam_40.ndpi
   → lam_41.ndpi
   → lam_42.ndpi
   → lam_43.ndpi
   → lam_44.ndpi
   → lam_45.ndpi
   → lam_46.ndpi
   → lam_47.ndpi
   → lam_48.ndpi
   → lam_49.ndpi
   → lam_50.ndpi
   → lam_51.ndpi
   → lam_52.ndpi
   → lam_53.ndpi
   → lam_54.ndpi
   → lam_55.ndpi
   → lam_56.ndpi
   → lam_57.ndpi
   → lam_58.ndpi
   → lam_59.ndpi
   → lam_60.ndpi
   → lam_61.ndpi


## Celda 3 — Extraer patches de todas las láminas

In [5]:
resumen = []

print('=' * 60)
print('  EXTRACCIÓN DE PATCHES — MÚLTIPLES LÁMINAS')
print('=' * 60)

for lamina_archivo in laminas:

    # Extraer número de lámina desde el nombre (lam_1.ndpi → 1)
    nro_lamina  = lamina_archivo.replace('lam_', '').replace('.ndpi', '')
    ruta_lamina = os.path.join(DIR_LAMINAS, lamina_archivo)

    # Directorio de salida para esta lámina
    dir_lamina = os.path.join(DIR_SALIDA, f'lamina_{nro_lamina}')
    os.makedirs(dir_lamina, exist_ok=True)

   # print(f'\n Procesando: {lamina_archivo}')
   # print(f'   Salida    : {dir_lamina}')

    try:
        # ── Abrir lámina ─────────────────────────────────────────────
        reader = WSIReader.open(ruta_lamina)
        info   = reader.info
      #  print(f'   Resolución base     : {info.mpp} µm/px')
       # print(f'   Magnificación base  : {info.objective_power}x')
      #  print(f'   Dimensiones nivel 0 : {info.slide_dimensions}')

        # ── Generar máscara de tejido (Otsu) ──────────────────────────
        masker    = OtsuTissueMasker()
        thumbnail = reader.slide_thumbnail(resolution=2.5, units='power')
        mask      = masker.fit_transform([thumbnail])[0]
        print(f'   Tejido detectado    : {mask.sum()} píxeles')

        # Guardar thumbnail y mask para verificación visual
        Image.fromarray(thumbnail).save(
            os.path.join(dir_lamina, f'thumbnail_lam_{nro_lamina}.png')
        )
        Image.fromarray((mask * 255).astype(np.uint8)).save(
            os.path.join(dir_lamina, f'mask_lam_{nro_lamina}.png')
        )

        # ── Extractor de patches ──────────────────────────────────────
        extractor = get_patch_extractor(
            input_img=reader,
            input_mask=mask,
            method_name='slidingwindow',
            patch_size=PATCH_SIZE,
            resolution=RESOLUTION,
            units=UNITS,
            stride=STRIDE,
            min_mask_ratio=MIN_MASK_RATIO,
        )

        # ── Guardar patches ───────────────────────────────────────────
        count = 0
        for patch in extractor:
            count += 1
            nombre_patch = f'pat_{count}_{nro_lamina}.png'
            Image.fromarray(patch).save(
                os.path.join(dir_lamina, nombre_patch),
                format='PNG',
                compress_level=PNG_COMPRESS
            )

            if count % 10 == 0:
                print(f'   Guardados: {count} patches...')

            if count >= MAX_PATCHES:
                print(f'   Límite alcanzado: {MAX_PATCHES} patches')
                break

        print(f'    Total patches guardados: {count}')

        # ── Verificar y borrar lámina .ndpi ───────────────────────────
        patches_guardados = [
            f for f in os.listdir(dir_lamina)
            if f.startswith('pat_') and f.endswith('.png')
        ]
        if len(patches_guardados) >= 1:
            os.remove(ruta_lamina)
            print(f'   🗑️  {lamina_archivo} eliminado definitivamente ({len(patches_guardados)} patches verificados)')
            estado_borrado = 'BORRADO'
        else:
            print(f'   ⚠️  {lamina_archivo} NO eliminado — no se encontraron patches en disco')
            estado_borrado = 'NO BORRADO'

        resumen.append({
            'lamina'    : lamina_archivo,
            'nro'       : nro_lamina,
            'patches'   : count,
            'directorio': dir_lamina,
            'estado'    : f'OK — {estado_borrado}'
        })

    except Exception as e:
        print(f'    ERROR: {e}')
        resumen.append({
            'lamina'    : lamina_archivo,
            'nro'       : nro_lamina,
            'patches'   : 0,
            'directorio': dir_lamina,
            'estado'    : f'ERROR: {e}'
        })

import pandas as pd

# ── Resumen final en consola ──────────────────────────────────────────────
print('\n' + '=' * 60)
print('  RESUMEN FINAL')
print('=' * 60)
total_patches = 0
for r in resumen:
    estado_icon = '✔' if r['estado'].startswith('OK') else '❌'
    print(f"{estado_icon} {r['lamina']:<20} → {r['patches']:>3} patches → lamina_{r['nro']}/ [{r['estado']}]")
    total_patches += r['patches']
print('─' * 60)
print(f'  Total de patches generados: {total_patches}')
print(f'  Total de láminas procesadas: {len(resumen)}')

# ── Exportar CSV de resultados ────────────────────────────────────────────
# Nombre del archivo: resultado_parches_<nro_ultima_lamina>.csv
ultima_lamina = resumen[-1]['nro'] if resumen else '0'
nombre_csv    = f'resultado_parches_{ultima_lamina}.csv'
ruta_csv      = os.path.join(DIR_SALIDA, nombre_csv)

df_resumen = pd.DataFrame([{
    'lamina'            : r['lamina'],
    'nro_lamina'        : r['nro'],
    'patches_generados' : r['patches'],
    'directorio_salida' : r['directorio'],
    'estado'            : r['estado'],
    'excepcion'         : r['estado'] if r['estado'].startswith('ERROR') else ''
} for r in resumen])

df_resumen.to_csv(ruta_csv, index=False, encoding='utf-8')
print(f'\n✔ CSV de resultados guardado: {ruta_csv}')

  EXTRACCIÓN DE PATCHES — MÚLTIPLES LÁMINAS
   Tejido detectado    : 22076010 píxeles
   Guardados: 10 patches...
   Guardados: 20 patches...
   Límite alcanzado: 20 patches
    Total patches guardados: 20
   🗑️  lam_36.ndpi eliminado definitivamente (20 patches verificados)
   Tejido detectado    : 24670128 píxeles
   Guardados: 10 patches...
   Guardados: 20 patches...
   Límite alcanzado: 20 patches
    Total patches guardados: 20
   🗑️  lam_37.ndpi eliminado definitivamente (20 patches verificados)
   Tejido detectado    : 30247383 píxeles
   Guardados: 10 patches...
   Guardados: 20 patches...
   Límite alcanzado: 20 patches
    Total patches guardados: 20
   🗑️  lam_38.ndpi eliminado definitivamente (20 patches verificados)
   Tejido detectado    : 11672796 píxeles
   Guardados: 10 patches...
   Guardados: 20 patches...
   Límite alcanzado: 20 patches
    Total patches guardados: 20
   🗑️  lam_39.ndpi eliminado definitivamente (20 patches verificados)
   Tejido detectado    : 269